In [ ]:
# Shared session setup (run in every notebook)
import os, subprocess, pathlib
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/s1pii'); DRIVE.mkdir(parents=True, exist_ok=True)
os.environ['DRIVE'] = str(DRIVE)
os.environ['S1PII_DATA'] = '/content/s1pii_data'        # local disk; synced to/from Drive
os.environ['S1PII_RESULTS'] = str(DRIVE / 'results')     # ledger + prediction cache on Drive
REPO = pathlib.Path('/content/s1-pii')

def git(*args):
    # The token (Colab secret GH_TOKEN: fine-grained, read-only on anitksahu/s1-pii) is passed
    # through GIT_ASKPASS, never on the command line, and errors are printed redacted.
    try:
        tok = (userdata.get('GH_TOKEN') or '').strip()
    except Exception:
        tok = ''
    if not tok:   # public repo: no token needed
        r = subprocess.run(['git', *args], capture_output=True, text=True)
        if r.returncode: raise RuntimeError('git ' + args[0] + ' failed: ' + (r.stderr or '').replace('https://x-access-token@github.com/anitksahu/s1-pii.git', '***')[-800:])
        return r.stdout
    askpass = pathlib.Path('/tmp/askpass.sh'); askpass.write_text('#!/bin/sh\necho "$S1PII_GIT_TOKEN"\n'); askpass.chmod(0o700)
    env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'S1PII_GIT_TOKEN': tok, 'GIT_TERMINAL_PROMPT': '0'}
    r = subprocess.run(['git', *args], env=env, capture_output=True, text=True)
    if r.returncode:
        safe = (r.stderr or '').replace(tok, '***').replace('https://x-access-token@github.com/anitksahu/s1-pii.git', '***')
        raise RuntimeError('git ' + args[0] + ' failed: ' + safe[-800:])
    return r.stdout

if not REPO.exists():
    git('clone', 'https://x-access-token@github.com/anitksahu/s1-pii.git', str(REPO))
git('-C', str(REPO), 'fetch')
git('-C', str(REPO), 'switch', 's1d')
git('-C', str(REPO), 'pull', '--ff-only')
os.chdir(REPO)
subprocess.run(['rsync', '-a', f'{DRIVE}/data/', os.environ['S1PII_DATA'] + '/'], check=False)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r envs/requirements-s1d.txt -e ".[data,model]" "faker==40.40.0"
!python -m pytest -q tests/test_s1d.py

In [ ]:
stage = 'stage0'
log_dir = DRIVE / 's1d' / 'logs'; log_dir.mkdir(parents=True, exist_ok=True)
log = open(log_dir / f'{stage}-launcher.log', 'a')
proc = subprocess.Popen(['bash', 'scripts/s1d_chain.sh', stage], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print('started', proc.pid, 'log', log.name)

In [ ]:
%run scripts/s1d_monitor.py